In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/sample_submission.csv
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/classes.csv
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/train.csv
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test.csv
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_00733.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_03144.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_01642.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_00121.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_00739.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_01420.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_01977.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_00432.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_02001.jpg
/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test/test_00180.jpg
/kaggle/input/competitions/nppe-1

In [2]:
# /kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/sample_submission.csv
# /kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/classes.csv
# /kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/train.csv
# /kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test.csv

train = pd.read_csv("/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/train.csv")
test = pd.read_csv("/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/sample_submission.csv")
classes = pd.read_csv("/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai/classes.csv")

In [3]:
classes.head()

,label,class_name,material,condition
0,0,glass_defect,glass,defect
1,1,glass_no_defect,glass,no_defect
2,2,rubber_defect,rubber,defect
3,3,rubber_no_defect,rubber,no_defect
4,4,hard_plastic_defect,hard_plastic,defect


In [4]:
# %% [1] Imports & config
import os, time, random
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DATA = "/kaggle/input/competitions/nppe-1-26-t-3-dl-gen-ai"   # check the exact folder name in the right sidebar
OUT = "/kaggle/working"

class CFG:
    img_size = 224        # try 256+ if defects are tiny and GPU time allows
    batch_size = 64
    epochs = 40
    lr = 2e-3
    wd = 5e-2
    label_smooth = 0.1
    val_frac = 0.15
    seed = 42
    workers = 4
    subset = None         # set e.g. 500 for a quick pipeline check, None for full run

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# %% [2] Load metadata
train_df = pd.read_csv(f"{DATA}/train.csv")
test_df = pd.read_csv(f"{DATA}/test.csv")
classes = pd.read_csv(f"{DATA}/classes.csv")
print(classes)
print(train_df.label.value_counts().sort_index())   # check class imbalance
NUM_CLASSES = classes.label.nunique()

if CFG.subset:
    train_df = train_df.sample(CFG.subset, random_state=CFG.seed).reset_index(drop=True)

tr_df, va_df = train_test_split(
    train_df, test_size=CFG.val_frac, stratify=train_df.label, random_state=CFG.seed)
print(len(tr_df), len(va_df), len(test_df))

# Peek at an image to confirm size
im = Image.open(f"{DATA}/train/{train_df.id[0]}.jpg"); print("sample image size:", im.size)

# %% [3] Dataset & transforms
# Mean/std: computed roughly; replace with real train stats if you like.
MEAN, STD = (0.5, 0.5, 0.5), (0.25, 0.25, 0.25)

class RandRot90:
    def __call__(self, img):
        k = random.randint(0, 3)
        return img.rotate(90 * k) if k else img

train_tf = T.Compose([
    T.Resize((CFG.img_size, CFG.img_size)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), RandRot90(),
    T.ColorJitter(0.2, 0.2, 0.2),            # mild; avoid heavy crops (could remove small defects)
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((CFG.img_size, CFG.img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, df, folder, tf, has_label=True):
        self.ids = df.id.values; self.folder = folder; self.tf = tf
        self.y = df.label.values if has_label else None
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = Image.open(f"{DATA}/{self.folder}/{self.ids[i]}.jpg").convert("RGB")
        x = self.tf(img)
        return (x, int(self.y[i])) if self.y is not None else x

def loader(ds, shuffle):
    return DataLoader(ds, batch_size=CFG.batch_size, shuffle=shuffle,
                      num_workers=CFG.workers, pin_memory=True, drop_last=shuffle)

train_dl = loader(DS(tr_df, "train", train_tf), True)
val_dl = loader(DS(va_df, "train", eval_tf), False)

# %% [4] Model: ResNet-style network, random init (NO pretrained weights)
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)), inplace=True)
        o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)), inplace=True)

class ResNetS(nn.Module):
    def __init__(self, n_cls, widths=(64, 128, 256, 512), depths=(2, 2, 2, 2)):
        super().__init__()
        # Gentle stem (stride 2, no maxpool) to preserve fine detail for small defects
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 2, 1, bias=False),
                                  nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, cin = [], widths[0]
        for i, (w, d) in enumerate(zip(widths, depths)):
            for j in range(d):
                layers.append(Block(cin, w, 2 if (j == 0 and i > 0) else 1)); cin = w
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(cin, n_cls)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(self.drop(F.adaptive_avg_pool2d(x, 1).flatten(1)))

model = ResNetS(NUM_CLASSES).to(device).to(memory_format=torch.channels_last)
print("params (M):", sum(p.numel() for p in model.parameters()) / 1e6)

# %% [5] Train
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smooth)
opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.lr, epochs=CFG.epochs,
                                            steps_per_epoch=len(train_dl), pct_start=0.2)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

@torch.no_grad()
def evaluate(dl):
    model.eval(); preds, ys = [], []
    for x, y in dl:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            preds.append(model(x).argmax(1).cpu())
        ys.append(y)
    p, y = torch.cat(preds).numpy(), torch.cat(ys).numpy()
    return f1_score(y, p, average="macro"), (p == y).mean()

best_f1 = 0
for ep in range(CFG.epochs):
    model.train(); t0 = time.time(); tot = 0
    for x, y in train_dl:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    f1, acc = evaluate(val_dl)
    if f1 > best_f1:
        best_f1 = f1; torch.save(model.state_dict(), f"{OUT}/best.pt")   # checkpoint your own trained weights
    print(f"ep {ep+1:02d}/{CFG.epochs} loss {tot/len(train_dl):.4f} val macroF1 {f1:.4f} acc {acc:.4f} "
          f"best {best_f1:.4f} ({time.time()-t0:.0f}s)")

# %% [6] Predict test with TTA and write submission.csv
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=device))
model.eval()
test_dl = loader(DS(test_df, "test", eval_tf, has_label=False), False)

@torch.no_grad()
def predict_tta(dl):
    out = []
    for x in dl:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            p = F.softmax(model(x), 1) + F.softmax(model(x.flip(3)), 1) + F.softmax(model(x.flip(2)), 1)
        out.append(p.cpu())
    return torch.cat(out).argmax(1).numpy()

sub = pd.DataFrame({"id": test_df.id, "label": predict_tta(test_dl)})
sub.to_csv(f"{OUT}/submission.csv", index=False)
print(sub.head(), sub.shape, sub.label.nunique())

device: cuda
    label                class_name        material  condition
0       0              glass_defect           glass     defect
1       1           glass_no_defect           glass  no_defect
2       2             rubber_defect          rubber     defect
3       3          rubber_no_defect          rubber  no_defect
4       4       hard_plastic_defect    hard_plastic     defect
5       5    hard_plastic_no_defect    hard_plastic  no_defect
6       6        wood_coated_defect     wood_coated     defect
7       7     wood_coated_no_defect     wood_coated  no_defect
8       8      wood_uncoated_defect   wood_uncoated     defect
9       9   wood_uncoated_no_defect   wood_uncoated  no_defect
10     10              cable_defect           cable     defect
11     11           cable_no_defect           cable  no_defect
12     12          cardboard_defect       cardboard     defect
13     13       cardboard_no_defect       cardboard  no_defect
14     14       metal_coated_defect    met

/tmp/ipykernel_23/3646207902.py:127: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))


ep 01/40 loss 2.8859 val macroF1 0.1227 acc 0.1553 best 0.1227 (32s)
ep 02/40 loss 2.8089 val macroF1 0.1141 acc 0.1567 best 0.1227 (27s)
ep 03/40 loss 2.7746 val macroF1 0.1368 acc 0.1675 best 0.1368 (29s)
ep 04/40 loss 2.7488 val macroF1 0.1123 acc 0.1372 best 0.1368 (30s)
ep 05/40 loss 2.7044 val macroF1 0.0755 acc 0.1271 best 0.1368 (29s)
ep 06/40 loss 2.6856 val macroF1 0.1310 acc 0.1762 best 0.1368 (29s)
ep 07/40 loss 2.6440 val macroF1 0.1441 acc 0.1829 best 0.1441 (30s)
ep 08/40 loss 2.5754 val macroF1 0.1424 acc 0.1742 best 0.1441 (29s)
ep 09/40 loss 2.5501 val macroF1 0.1592 acc 0.1863 best 0.1592 (29s)
ep 10/40 loss 2.4910 val macroF1 0.1727 acc 0.2186 best 0.1727 (29s)
ep 11/40 loss 2.4320 val macroF1 0.1949 acc 0.2286 best 0.1949 (29s)
ep 12/40 loss 2.3970 val macroF1 0.2445 acc 0.2710 best 0.2445 (29s)
ep 13/40 loss 2.3662 val macroF1 0.2304 acc 0.2502 best 0.2445 (29s)
ep 14/40 loss 2.3104 val macroF1 0.2444 acc 0.2811 best 0.2445 (29s)
ep 15/40 loss 2.2900 val macroF1 0